# SITS - Treinamento de Modelos

Este notebook treina modelos de classificacao de series temporais.

**Entrada:**
- `dataset.npz`: Arrays numpy (X, y) gerados pelo notebook 03
- `dataset_metadata.json`: Mapeamento de classes

**Saida:** Modelos treinados na pasta `sessions/<projeto>/training/<experimento>/`

**Modelos disponiveis:**
- CNNs: InceptionTime, ResNet, FCN, TCN, XCM
- RNNs: LSTM, GRU (com variantes attention e bidirectional)
- Transformers: TST, TSTPlus, TSiTPlus, ConvTranPlus
- Hibridos: TransformerLSTM, TransformerGRU

## 1. Setup

In [1]:
import sys
sys.path.insert(0, '..')

import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch

from IPython.display import display, clear_output

# Device & seeds
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True

print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Device: cuda
GPU: NVIDIA GeForce RTX 4090


In [2]:
# Imports do modulo de treinamento
from sits.classification import ExperimentManager

---
## 2. Configuracao do Experimento

In [3]:
# =============================================================================
# CONFIGURACAO - EDITE AQUI
# =============================================================================

# Caminho para a sessao
SESSION_PATH = "../sessions/seu_projeto"

# Arquivos de entrada (gerados pelo notebook 03)
DATASET_FILE = Path(SESSION_PATH) / "annotation" / "dataset.npz"
CLASS_MAPPING_FILE = Path(SESSION_PATH) / "annotation" / "dataset_metadata.json"

# Nome do experimento
EXPERIMENT_NAME = "exp_v1"

# Descricao
DESCRIPTION = "Primeiro experimento de classificacao"

# -----------------------------------------------------------------------------
# Recorte temporal (opcional)
# -----------------------------------------------------------------------------
# None = usar todos os timesteps
# (3, -3) = do timestep 3 ate o penultimo 3
# (0, 6) = primeiros 6 timesteps
TIME_RANGE = None

# -----------------------------------------------------------------------------
# Split dos dados
# -----------------------------------------------------------------------------
TRAIN_SIZE = 0.6
VAL_SIZE = 0.2
TEST_SIZE = 0.2

# -----------------------------------------------------------------------------
# Parametros de treinamento
# -----------------------------------------------------------------------------
BATCH_SIZE = 64
NUM_EPOCHS = 1000
LEARNING_RATE = 1e-4
EARLY_STOP = 100  # Parar se nao melhorar por N epocas

# -----------------------------------------------------------------------------
# Forcar re-treinamento?
# -----------------------------------------------------------------------------
FORCE_RETRAIN = False  # True para re-treinar modelos existentes

## 3. Verificar Dados

In [4]:
# Verificar arquivos de entrada
print("Verificando arquivos de entrada:")
print("-" * 50)

files_ok = True

if DATASET_FILE.exists():
    size = DATASET_FILE.stat().st_size / 1024 / 1024
    print(f"  [OK] Dataset: {DATASET_FILE.name} ({size:.2f} MB)")
else:
    print(f"  [!!] Dataset NAO ENCONTRADO: {DATASET_FILE}")
    files_ok = False

if CLASS_MAPPING_FILE.exists():
    size = CLASS_MAPPING_FILE.stat().st_size / 1024
    print(f"  [OK] Metadata: {CLASS_MAPPING_FILE.name} ({size:.1f} KB)")
else:
    print(f"  [!!] Metadata NAO ENCONTRADO: {CLASS_MAPPING_FILE}")
    files_ok = False

if not files_ok:
    print("\nExecute o notebook 03_merge_dados.ipynb primeiro.")

Verificando arquivos de entrada:
--------------------------------------------------
  [!!] Dataset NAO ENCONTRADO: ..\sessions\seu_projeto\annotation\dataset.npz
  [!!] Metadata NAO ENCONTRADO: ..\sessions\seu_projeto\annotation\dataset_metadata.json

Execute o notebook 03_merge_dados.ipynb primeiro.


In [5]:
# Carregar e analisar dados
if files_ok:
    # Carregar NPZ
    data = np.load(DATASET_FILE)
    X = data["X"]
    y = data["y"]
    
    # Carregar metadata
    with open(CLASS_MAPPING_FILE, "r", encoding="utf-8") as f:
        metadata = json.load(f)
    
    class_mapping = metadata.get("class_mapping", {})
    idx_to_name = {int(k): v for k, v in metadata.get("idx_to_name", {}).items()}
    band_names = metadata.get("band_names", [])
    
    print(f"Dataset: {DATASET_FILE.name}")
    print(f"X shape: {X.shape}  (N, C, T)")
    print(f"y shape: {y.shape}  (N,)")
    print(f"Bandas: {band_names}")
    print()
    print("Distribuicao por classe:")
    print("-" * 50)
    total = 0
    for idx in sorted(idx_to_name.keys()):
        name = idx_to_name[idx]
        count = int(np.sum(y == idx))
        pct = count / len(y) * 100
        print(f"  {idx}: {name:20s}: {count:6d} ({pct:5.1f}%)")
        total += count
    print("-" * 50)
    print(f"     {'TOTAL':20s}: {total:6d}")
    print()
    print(f"Numero de classes: {len(class_mapping)}")

---
## 4. Selecao de Modelos

Comente/descomente as linhas para selecionar quais modelos treinar.

In [7]:
# =============================================================================
# LISTA DE MODELOS
# =============================================================================

MODELS_TO_TRAIN = [
    # -------------------------------------------------------------------------
    # CNNs 1D - InceptionTime (recomendado)
    # -------------------------------------------------------------------------
    "inception_time__default",
     "inception_time_plus__default",

    # -------------------------------------------------------------------------
    # CNNs 1D - ResNet
    # -------------------------------------------------------------------------
    "resnet__default",
     "resnet_plus__default",

    # -------------------------------------------------------------------------
    # CNNs 1D - FCN
    # -------------------------------------------------------------------------
    "fcn__default",
     "fcn_plus__default",

    # -------------------------------------------------------------------------
    # TCN (Temporal Convolutional Network)
    # -------------------------------------------------------------------------
     "tcn__default",

    # -------------------------------------------------------------------------
    # XCM
    # -------------------------------------------------------------------------
     "xcm__default",
     "xcm_plus__default",

    # -------------------------------------------------------------------------
    # ConvTranPlus (Convolutional Transformer)
    # -------------------------------------------------------------------------
     "conv_tran_plus__default",
     "conv_tran_plus__dim_ff_512",

    # -------------------------------------------------------------------------
    # LSTM
    # -------------------------------------------------------------------------
    "lstm__bidir",
     "lstm__unidir",
     "lstm_plus__bidir",
     "lstm_attention__bidir",

    # -------------------------------------------------------------------------
    # GRU
    # -------------------------------------------------------------------------
    "gru__bidir",
     "gru__unidir",
     "gru_plus__bidir",
     "gru_attention__bidir",

    # -------------------------------------------------------------------------
    # Transformers - TST
    # -------------------------------------------------------------------------
     "tst__default",
     "tst__h8",
     "tst_plus__default",

    # -------------------------------------------------------------------------
    # Transformers - TSiTPlus
    # -------------------------------------------------------------------------
     "tsit_plus__default",

    # -------------------------------------------------------------------------
    # Hibridos Transformer-RNN
    # -------------------------------------------------------------------------
     "transformer_lstm_plus__bidir",
     "transformer_gru_plus__bidir",

    # -------------------------------------------------------------------------
    # TSPerceiver
    # -------------------------------------------------------------------------
     "ts_perceiver__default",
]

print(f"Modelos selecionados: {len(MODELS_TO_TRAIN)}")
for m in MODELS_TO_TRAIN:
    print(f"  - {m}")

Modelos selecionados: 26
  - inception_time__default
  - inception_time_plus__default
  - resnet__default
  - resnet_plus__default
  - fcn__default
  - fcn_plus__default
  - tcn__default
  - xcm__default
  - xcm_plus__default
  - conv_tran_plus__default
  - conv_tran_plus__dim_ff_512
  - lstm__bidir
  - lstm__unidir
  - lstm_plus__bidir
  - lstm_attention__bidir
  - gru__bidir
  - gru__unidir
  - gru_plus__bidir
  - gru_attention__bidir
  - tst__default
  - tst__h8
  - tst_plus__default
  - tsit_plus__default
  - transformer_lstm_plus__bidir
  - transformer_gru_plus__bidir
  - ts_perceiver__default


In [8]:
# =============================================================================
# HIPERPARAMETROS POR MODELO
# =============================================================================

# RNNs: bidirectional ou unidirectional
RNN_CONFIGS = {
    "bidir": {"bidirectional": True},
    "unidir": {"bidirectional": False},
}

# Transformers: variacoes de n_heads e dropout
TRANSFORMER_CONFIGS = {
    "default": {},
    "h12": {"n_heads": 12, "d_model": 144},
    "h8": {"n_heads": 8},
    "drop05": {"dropout": 0.5},
}

# ConvTranPlus
CONV_TRAN_CONFIGS = {
    "default": {},
    "encoder_learned": {"abs_pos_encode": "learned"},
    "dim_ff_512": {"dim_ff": 512},
}

# TSPerceiver
PERCEIVER_CONFIGS = {
    "default": {},
    "8_heads": {"self_n_heads": 8},
}

# CNNs e outros
DEFAULT_CONFIG = {
    "default": {},
}

# Mapeamento modelo -> configs
MODEL_HYPERPARAMS = {
    # RNNs
    "lstm": RNN_CONFIGS,
    "lstm_plus": RNN_CONFIGS,
    "lstm_attention": RNN_CONFIGS,
    "gru": RNN_CONFIGS,
    "gru_plus": RNN_CONFIGS,
    "gru_attention": RNN_CONFIGS,
    "transformer_gru_plus": RNN_CONFIGS,
    "transformer_lstm_plus": RNN_CONFIGS,
    "transformer_rnn_plus": RNN_CONFIGS,
    
    # Transformers
    "tst": TRANSFORMER_CONFIGS,
    "tst_plus": TRANSFORMER_CONFIGS,
    "tsit_plus": TRANSFORMER_CONFIGS,
    
    # ConvTranPlus
    "conv_tran_plus": CONV_TRAN_CONFIGS,
    
    # TSPerceiver
    "ts_perceiver": PERCEIVER_CONFIGS,
    
    # CNNs
    "inception_time": DEFAULT_CONFIG,
    "inception_time_plus": DEFAULT_CONFIG,
    "resnet": DEFAULT_CONFIG,
    "resnet_plus": DEFAULT_CONFIG,
    "fcn": DEFAULT_CONFIG,
    "fcn_plus": DEFAULT_CONFIG,
    "tcn": DEFAULT_CONFIG,
    "xcm": DEFAULT_CONFIG,
    "xcm_plus": DEFAULT_CONFIG,
}

---
## 5. Criar Experimento e Preparar Dados

In [9]:
# Criar ou carregar experimento
experiment_dir = Path(SESSION_PATH) / "training" / EXPERIMENT_NAME

if experiment_dir.exists():
    print(f"Carregando experimento existente: {experiment_dir}")
    exp = ExperimentManager.load(experiment_dir)
else:
    print(f"Criando novo experimento: {EXPERIMENT_NAME}")
    exp = ExperimentManager.create(
        session_path=SESSION_PATH,
        name=EXPERIMENT_NAME,
        description=DESCRIPTION,
        time_range=TIME_RANGE,
    )

Criando novo experimento: exp_v1


In [10]:
# Preparar dados (se ainda nao preparados)
splits_file = exp.data_dir / "splits.npz"

if not splits_file.exists():
    print("Preparando dados...")
    
    data_info = exp.prepare_data_from_npz(
        npz_path=DATASET_FILE,
        class_mapping_path=CLASS_MAPPING_FILE,
        train=TRAIN_SIZE,
        val=VAL_SIZE,
        test=TEST_SIZE,
        seed=SEED,
        time_range=TIME_RANGE,
    )
else:
    print(f"Dados ja preparados: {splits_file}")
    splits = np.load(splits_file)
    print(f"  Train: {len(splits['y_train'])} samples")
    print(f"  Val: {len(splits['y_val'])} samples")
    print(f"  Test: {len(splits['y_test'])} samples")
    print(f"  Shape: {splits['X_train'].shape[1]} channels, {splits['X_train'].shape[2]} timesteps")

Preparando dados...


FileNotFoundError: [Errno 2] No such file or directory: '..\\sessions\\seu_projeto\\annotation\\dataset.npz'

---
## 6. Treinar Modelos

In [ ]:
# Verificar quais modelos ja foram treinados
def is_model_trained(model_id):
    model_dir = exp.models_dir / model_id
    return (model_dir / "model.pt").exists()

if FORCE_RETRAIN:
    models_to_run = MODELS_TO_TRAIN
    print(f"FORCE_RETRAIN=True: Treinando todos os {len(models_to_run)} modelos")
else:
    models_to_run = [m for m in MODELS_TO_TRAIN if not is_model_trained(m)]
    already_trained = len(MODELS_TO_TRAIN) - len(models_to_run)
    print(f"Modelos ja treinados: {already_trained}")
    print(f"Modelos a treinar: {len(models_to_run)}")

if models_to_run:
    print("\nModelos pendentes:")
    for m in models_to_run:
        print(f"  - {m}")

In [ ]:
# === Executar treinamento ===
results = []

for i, model_id in enumerate(models_to_run):
    # Parse model_name e variant
    parts = model_id.split("__")
    model_name = parts[0]
    variant = parts[1] if len(parts) > 1 else "default"
    
    # Get hyperparameters for this variant
    model_configs = MODEL_HYPERPARAMS.get(model_name, DEFAULT_CONFIG)
    kwargs = model_configs.get(variant, {})
    
    print(f"\n{'='*60}")
    print(f"[{i+1}/{len(models_to_run)}] Treinando: {model_id}")
    print(f"{'='*60}")
    
    try:
        metrics = exp.train_model(
            model_name=model_name,
            variant=variant,
            batch_size=BATCH_SIZE,
            epochs=NUM_EPOCHS,
            learning_rate=LEARNING_RATE,
            early_stop=EARLY_STOP,
            force_retrain=FORCE_RETRAIN,
            verbose=True,
            **kwargs,
        )
        
        results.append({
            "model": model_id,
            "accuracy": metrics["accuracy"],
            "f1_score": metrics["f1_score"],
        })
        
        # Mostrar progresso
        clear_output(wait=True)
        df_progress = pd.DataFrame(results)
        df_progress = df_progress.sort_values("accuracy", ascending=False).reset_index(drop=True)
        print(f"Progresso: {i+1}/{len(models_to_run)} modelos treinados")
        print()
        display(df_progress)
        
    except Exception as e:
        print(f"Erro ao treinar {model_id}: {e}")
        results.append({
            "model": model_id,
            "accuracy": None,
            "f1_score": None,
            "error": str(e),
        })

print("\n" + "="*60)
print("TREINAMENTO CONCLUIDO!")
print("="*60)

---
## 7. Resultados

In [ ]:
# Resumo de todos os modelos treinados
summary = exp.summary()
print(f"Total de modelos treinados: {len(summary)}")
print()
display(summary)

In [ ]:
# Salvar resumo em CSV
summary_path = exp.experiment_dir / "summary.csv"
summary.to_csv(summary_path, index=False)
print(f"Resumo salvo em: {summary_path}")

In [ ]:
# Melhor modelo
if len(summary) > 0:
    best_name, best_path = exp.get_best_model()
    best_row = summary[summary["model"] == best_name].iloc[0]
    
    print(f"MELHOR MODELO: {best_name}")
    print(f"  Accuracy: {best_row['accuracy']:.4f}")
    print(f"  F1-Score: {best_row['f1_score']:.4f}")
    print(f"  Path: {best_path}")

In [ ]:
# Visualizar top modelos
if len(summary) > 0:
    n_show = min(10, len(summary))
    top = summary.head(n_show)
    
    fig, ax = plt.subplots(figsize=(12, max(4, n_show * 0.5)))
    bars = ax.barh(top["model"], top["accuracy"], color="steelblue")
    ax.set_xlabel("Accuracy")
    ax.set_title(f"Top {n_show} Modelos - {EXPERIMENT_NAME}")
    ax.invert_yaxis()
    
    # Adicionar valores
    for bar, acc in zip(bars, top["accuracy"]):
        ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
                f"{acc:.4f}", va="center", fontsize=9)
    
    plt.tight_layout()
    plt.savefig(exp.experiment_dir / "top_models.png", dpi=150)
    plt.show()

In [ ]:
# Comparacao por familia de modelo
if len(summary) > 0:
    # Extrair familia do modelo
    summary_copy = summary.copy()
    summary_copy["family"] = summary_copy["model"].apply(
        lambda x: x.split("__")[0].replace("_plus", "").replace("_attention", "")
    )
    
    # Melhor de cada familia
    best_per_family = summary_copy.groupby("family").first().reset_index()
    best_per_family = best_per_family[["family", "model", "accuracy", "f1_score"]]
    best_per_family = best_per_family.sort_values("accuracy", ascending=False)
    
    print("Melhor modelo de cada familia:")
    print("-" * 60)
    display(best_per_family)

---
## Proximo Passo

Execute o notebook **05_classificacao.ipynb** para aplicar o melhor modelo em imagens de serie temporal.